In [2]:
import ollama
import json
import math

# --- Helper Functions ---
def dot_product(v1, v2):
    return sum(a * b for a, b in zip(v1, v2))

def magnitude(v):
    return math.sqrt(sum(a * a for a in v))

def cosine_similarity(v1, v2):
    return dot_product(v1, v2) / (magnitude(v1) * magnitude(v2))

# --- 1. Read and chunk your log files ---
def load_log_file(filepath, max_lines=500):
    chunks = []
    with open(filepath, 'r', errors='ignore') as f:  # errors='ignore' handles binary garbage in logs
        lines = f.readlines()
    
    # Group lines into chunks of 5 (so each vector covers a few related lines)
    chunk_size = 5
    for i in range(0, len(lines), chunk_size):
        chunk = "".join(lines[i:i+chunk_size]).strip()
        if chunk:  # skip empty chunks
            chunks.append(chunk)
    
    return chunks[:max_lines]  # cap it to avoid embedding forever

# --- 2. Embed and store ---
log_files = [
    "/var/log/auth.log",
]

data_to_store = []
for log_path in log_files:
    print(f"Processing {log_path}...")
    chunks = load_log_file(log_path)
    for chunk in chunks:
        response = ollama.embed(model='nomic-embed-text', input=chunk)
        vector = response['embeddings'][0]
        data_to_store.append({
            "text": chunk,
            "source": log_path,  # track which file it came from
            "vector": vector
        })
    print(f"  → {len(chunks)} chunks embedded from {log_path}")

with open("log_vectors.json", "w") as f:
    json.dump(data_to_store, f)

print(f"\nDone! Saved {len(data_to_store)} chunks to log_vectors.json")

# --- 3. Search ---
def search(query, database, top_k=3):
    response = ollama.embed(model='nomic-embed-text', input=query)
    query_vector = response['embeddings'][0]
    
    results = []
    for entry in database:
        score = cosine_similarity(query_vector, entry["vector"])
        results.append((score, entry["source"], entry["text"]))
    
    results.sort(reverse=True)
    return results[:top_k]

# --- 4. Load and query ---
with open("log_vectors.json", "r") as f:
    database = json.load(f)

query = "Accepted publickey for user from "
print(f"\nSearching for: '{query}'\n")
top_results = search(query, database)

for i, (score, source, text) in enumerate(top_results, 1):
    print(f"Result #{i} (score: {score:.4f}) from {source}")
    print(f"{text}")
    print("---")

Processing /var/log/auth.log...
  → 90 chunks embedded from /var/log/auth.log

Done! Saved 90 chunks to log_vectors.json

Searching for: 'Accepted publickey for user from '

Result #1 (score: 0.5973) from /var/log/auth.log
2026-02-23T12:35:01.152550+00:00 noble CRON[263744]: pam_unix(cron:session): session closed for user root
2026-02-23T12:38:30.300154+00:00 noble sshd[265379]: Server listening on 0.0.0.0 port 22.
2026-02-23T12:38:30.300299+00:00 noble sshd[265379]: Server listening on :: port 22.
2026-02-23T12:38:30.454984+00:00 noble sshd[265381]: Accepted publickey for ubuntu from 127.0.0.1 port 58572 ssh2: RSA SHA256:MsNUHzdQxHLJ+MU/wlNW/gLQAN5/s4WfRSLDLF1MpEQ
2026-02-23T12:38:30.458682+00:00 noble sshd[265381]: pam_unix(sshd:session): session opened for user ubuntu(uid=1000) by ubuntu(uid=0)
---
Result #2 (score: 0.5941) from /var/log/auth.log
2026-02-22T21:32:10.487415+00:00 noble gnome-keyring-daemon[19432]: asked to register item /org/freedesktop/secrets/collection/login/1, bu